In [3]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from shapely.geometry import Point, Polygon, LineString, box
from shapely.ops import unary_union, nearest_points
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set up paths
WATER_MASK_DIR = 'dataset/baringo/processed/sar_water_mask'
OUTPUT_DIR = 'dataset/baringo/processed/expansion_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/figures', exist_ok=True)

# Display settings
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

print("✅ Environment setup complete")
print(f"📁 Water masks directory: {WATER_MASK_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")

KeyboardInterrupt: 

In [ ]:
from src.acquisition.roads_acquisition import RoadsAcquisition
from config import get_roi_by_name

roi = get_roi_by_name("bogoria")
RoadsAcquisition(region="bogoria").acquire_roads(roi)

## Load and Catalogue Water Masks

We scan the directory for all available water masks, extract their dates from filenames,
and organise them chronologically. Each mask is a binary GeoTIFF where:
- **1** = Water
- **0** = Non-water

In [ ]:
kenya_lakes_gdf = gpd.GeoDataFrame(
    lakes, 
    geometry=lakes['geometry'],  # or the column containing geometry
    crs=gdf.crs  # preserve the CRS
)

# Step 5: Save as shapefile
kenya_lakes_gdf.to_file('/home/desy/rift-waters/dataset/bounderies/lakes/kenya_lakes.shp')

In [ ]:
import geopandas as gpd
import ee

# Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='riftwaters')
# Load your shapefile
kenya_lakes = gpd.read_file('kenya_lakes.shp')

# Convert GeoDataFrame to Earth Engine FeatureCollection
def gdf_to_ee(gdf):
    """Convert GeoDataFrame to Earth Engine FeatureCollection"""
    features = []
    
    for idx, row in gdf.iterrows():
        # Get geometry as GeoJSON
        geom = row.geometry.__geo_interface__
        
        # Get properties (attributes)
        props = row.drop('geometry').to_dict()
        
        # Create EE Feature
        feature = ee.Feature(geom, props)
        features.append(feature)
    
    return ee.FeatureCollection(features)

# Convert to EE FeatureCollection
ee_lakes = gdf_to_ee(kenya_lakes)

# Upload to Earth Engine assets
asset_id = 'projects/riftwaters/assets/kenya_lakes'  # Change this to your path
task = ee.batch.Export.table.toAsset(
    collection=ee_lakes,
    description='Upload_Kenya_Lakes',
    assetId=asset_id
)
task.start()

# Monitor upload progress
print(f"Upload started. Check Earth Engine Tasks tab: {task.status()}")

Upload started. Check Earth Engine Tasks tab: {'state': 'READY', 'description': 'Upload_Kenya_Lakes', 'priority': 100, 'creation_timestamp_ms': 1783148663555, 'update_timestamp_ms': 1783148663555, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': '7E54X2DJLQJM4TZEAOX6VE54', 'name': 'projects/riftwaters/operations/7E54X2DJLQJM4TZEAOX6VE54'}


In [ ]:
"""
Stage 4: Extract lake boundary from a Sentinel-1 water mask (GEE)

Assumes you already have from stages 1-3:
    water_mask : ee.Image  -- binary image, 1 = water, 0 = non-water
    aoi        : ee.Geometry or ee.FeatureCollection -- your study area
    scale      : pixel resolution of your Sentinel-1 data (typically 10m)

Plug your existing objects in where marked below.
"""

import ee
# ee.Initialize(project='riftwaters')  # uncomment if not already initialized


ee.Authenticate()
ee.Initialize(project='riftwaters')

# ---- 0. Inputs from your existing stage 1-3 code ----
water_mask = "/home/desy/rift-waters/dataset/bogoria/processed/sar_water_mask/bogoria_20161031.tif"   # <- replace with your actual variable
aoi = ee.Geometry.Polygon(
            [
                [36.021644575238554, 0.15353594768753728],
                [36.18025968754324, 0.15353594768753728],
                [36.18025968754324, 0.36364721537653627],
                [36.021644575238554, 0.36364721537653627],
                [36.021644575238554, 0.15353594768753728],
            ])                 # <- replace with your actual variable
scale = 10
"""
Stage 4: Extract lake boundary from a saved SAR water mask (.tif)

Fits into your existing SARProcessor class -- add extract_lake_boundary()
as a method, or use it standalone by passing a water_mask_path.

Requires: geopandas, shapely (rasterio, numpy already in your stack)
    pip install geopandas shapely
"""

import os
from pathlib import Path

import geopandas as gpd
import rasterio
import rasterio.features
from shapely.geometry import shape


"""
Stage 4: Extract lake boundary from a saved SAR water mask (.tif)

Fits into your existing SARProcessor class -- add extract_lake_boundary()
as a method, or use it standalone by passing a water_mask_path.

Requires: geopandas, shapely (rasterio, numpy already in your stack)
    pip install geopandas shapely
"""

import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
import rasterio.features
from shapely.geometry import shape


def extract_lake_boundary(
    region,
    image_id,
    water_mask_path=None,
    min_area_m2=10000,
    simplify_tolerance=10,
):
    """
    Vectorize a saved binary water mask GeoTIFF into cleaned lake boundary
    polygons and save as a shapefile.

    Parameters
    ----------
    region : str
        Same region name used in SARProcessor (for path construction).
    image_id : str
        Same image_id used in SARProcessor.
    water_mask_path : str or Path, optional
        Path to the water mask .tif. Defaults to where SARProcessor saves it:
        dataset/{region}/processed/sar_water_mask/{region}_{image_id}.tif
    min_area_m2 : float
        Drops polygons smaller than this (filters speckle-driven noise).
        Tune down if your lake has thin bays/inlets you want to keep.
    simplify_tolerance : float
        Simplification tolerance in map units (meters, if CRS is projected).
        ~10m matches Sentinel-1's pixel size, so it won't erase real shoreline detail.

    Returns
    -------
    geopandas.GeoDataFrame or None
    """
    if water_mask_path is None:
        water_mask_path = Path(
            f"../dataset/{region}/processed/sar_water_mask/{region}_{image_id}.tif"
        )

    with rasterio.open(water_mask_path) as src:
        water_mask = src.read(1)
        transform = src.transform
        crs = src.crs

    # Vectorize: rasterio.features.shapes groups connected pixels of the same
    # value into polygons. We only keep groups where value == 1 (water).
    shapes_gen = rasterio.features.shapes(
        water_mask.astype("uint8"), transform=transform
    )
    geometries = [shape(geom) for geom, value in shapes_gen if value == 1]

    if not geometries:
        print(f"  ⚠️ No water polygons found for {region}_{image_id}")
        return None

    gdf = gpd.GeoDataFrame({"geometry": geometries}, crs=crs)

    # If your CRS is geographic (e.g. EPSG:4326), area is in degrees^2 and
    # meaningless -- reproject to a local UTM zone first. Skip this if your
    # water mask is already in a projected CRS (which it should be, coming
    # straight from Sentinel-1 GRD in UTM).
    if gdf.crs is not None and gdf.crs.is_geographic:
        gdf = gdf.to_crs(gdf.estimate_utm_crs())

    gdf["area_m2"] = gdf.geometry.area

    original_count = len(gdf)
    gdf = gdf[gdf["area_m2"] >= min_area_m2].reset_index(drop=True)
    print(f"  Boundary cleaning: {original_count} -> {len(gdf)} polygons")

    gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance)

    # Recompute area post-simplify (simplify can shift it slightly), then
    # derive perimeter and a compactness index. Compactness = 1 means a
    # perfect circle; lower values mean a more convoluted/jagged shoreline.
    # Tracking this alongside area helps distinguish genuine outward
    # expansion from a shoreline just becoming more irregular.
    gdf["area_m2"] = gdf.geometry.area
    gdf["perimeter_m"] = gdf.geometry.length
    gdf["compactness"] = (4 * np.pi * gdf["area_m2"]) / (gdf["perimeter_m"] ** 2)

    output_dir = f"dataset/{region}/processed/lake_boundaries/"
    os.makedirs(output_dir, exist_ok=True)
    shp_path = Path(output_dir) / f"{region}_{image_id}_boundary.shp"
    gdf.to_file(shp_path)
    print(f"  💾 Saved lake boundary shapefile to {shp_path}")
    print(
        f"  📏 Total area: {gdf['area_m2'].sum():,.0f} m² | "
        f"Total perimeter: {gdf['perimeter_m'].sum():,.0f} m | "
        f"Mean compactness: {gdf['compactness'].mean():.2f}"
    )

    return gdf


# --- Example usage ---
gdf = extract_lake_boundary(region="bogoria", image_id="20161031")

  Boundary cleaning: 1 -> 1 polygons
  💾 Saved lake boundary shapefile to dataset/bogoria/processed/lake_boundaries/bogoria_20161031_boundary.shp
  📏 Total area: 39,654,530 m² | Total perimeter: 63,987 m | Mean compactness: 0.12


In [ ]:
"""
DEMAcquisition: downloads Copernicus GLO-30 DEM tiles via Google Earth Engine
for a given region/AOI, saved locally as GeoTIFF -- matches the folder
structure and metadata-logging style of SARProcessor.

Requires: earthengine-api, geemap
    pip install earthengine-api geemap
"""

import ee
import geemap
import json
import os
import rasterio
from pathlib import Path
from datetime import datetime
# ee.Initialize(project='riftwaters')  # uncomment if not already initialized


ee.Authenticate()
ee.Initialize(project='riftwaters')

# ---- 0. Inputs from your existing stage 1-3 code ----
water_mask = "/home/desy/rift-waters/dataset/bogoria/processed/sar_water_mask/bogoria_20161031.tif"   # <- replace with your actual variable
aoi = ee.Geometry.Polygon(
            [
                [36.021644575238554, 0.15353594768753728],
                [36.18025968754324, 0.15353594768753728],
                [36.18025968754324, 0.36364721537653627],
                [36.021644575238554, 0.36364721537653627],
                [36.021644575238554, 0.15353594768753728],
            ])                 # <- replace with your actual variable
scale = 10

class DEMAcquisition:
    def __init__(self, region, aoi):
        """
        Parameters
        ----------
        region : str
            Region name, used for folder naming (matches SARProcessor).
        aoi : ee.Geometry or ee.FeatureCollection
            Area of interest to clip/export the DEM to.
        project_id : str
            GEE cloud project id for ee.Initialize().
        """
        self.region = region
        self.aoi = aoi
        # self.project_id = project_id
        # ee.Initialize(project=project_id)

    def get_copernicus_dem(self):
        """
        Loads the Copernicus GLO-30 DEM collection, mosaics tiles covering
        the AOI, and clips to it. GLO-30 is distributed as an
        ImageCollection of individual tiles, so mosaic() is required even
        for a single-tile AOI -- this also handles AOIs that straddle
        tile boundaries.
        """
        dem_collection = ee.ImageCollection("COPERNICUS/DEM/GLO30")
        dem = dem_collection.select("DEM").mosaic().clip(self.aoi)
        return dem

    def download_dem(self, scale=30, crs="EPSG:4326", max_direct_mb=50):
        """
        Downloads the DEM as a local GeoTIFF.

        For small AOIs, uses geemap's direct download (fast, synchronous).
        For larger AOIs this will hit GEE's direct-download size limit --
        in that case, use export_dem_to_drive() instead, which submits a
        batch task with no size cap.

        Parameters
        ----------
        scale : int
            Output resolution in meters. 30 matches GLO-30's native
            resolution -- don't go finer, you'd just be interpolating.
        crs : str
            Output CRS. EPSG:4326 (lat/lon) is GLO-30's native CRS.
            Reproject to a local UTM zone if you need this to line up
            in meters with your Sentinel-1-derived lake boundaries for
            volume calculations.
        max_direct_mb : float
            Rough safety warning threshold before attempting direct
            download (GEE's actual hard limit is ~50MB for
            geemap.ee_export_image's default request).

        Returns
        -------
        Path to the downloaded .tif
        """
        dem = self.get_copernicus_dem()

        output_dir = f"dataset/{self.region}/raw/dem/"
        os.makedirs(output_dir, exist_ok=True)
        output_path = Path(output_dir) / f"{self.region}_dem_glo30.tif"

        print(f"  ⬇️ Downloading Copernicus GLO-30 DEM for {self.region}...")
        try:
            geemap.ee_export_image(
                dem,
                filename=str(output_path),
                scale=scale,
                region=self.aoi,
                crs=crs,
                file_per_band=False,
            )
        except Exception as e:
            print(f"  ❌ Direct download failed (AOI may be too large): {e}")
            print(
                "  💡 Try export_dem_to_drive() instead -- no size limit, "
                "but writes to Drive asynchronously as a batch task."
            )
            return None

        print(f"  ✅ Saved DEM to {output_path}")
        self._save_metadata(output_path, scale, crs, method="direct_download")
        return output_path

    def export_dem_to_drive(self, scale=30, crs="EPSG:4326", folder="GEE_exports"):
        """
        Submits a GEE batch export task to write the DEM to Google Drive.
        Use this instead of download_dem() for AOIs too large for direct
        download. Unlike download_dem(), this is asynchronous -- check the
        GEE Tasks tab or Code Editor console for completion, then download
        the file from Drive manually into
        dataset/{region}/raw/dem/{region}_dem_glo30.tif to keep your
        folder structure consistent.
        """
        dem = self.get_copernicus_dem()

        task = ee.batch.Export.image.toDrive(
            image=dem,
            description=f"{self.region}_dem_glo30_export",
            folder=folder,
            fileNamePrefix=f"{self.region}_dem_glo30",
            region=self.aoi,
            scale=scale,
            crs=crs,
            maxPixels=1e10,
        )
        task.start()
        print(
            f"  🚀 Export task started: {self.region}_dem_glo30_export "
            f"(check GEE Tasks tab for progress)"
        )
        self._save_metadata(
            output_path=f"Drive:{folder}/{self.region}_dem_glo30.tif",
            scale=scale,
            crs=crs,
            method="drive_export_task",
        )
        return task

    def _save_metadata(self, output_path, scale, crs, method):
        """Logs DEM acquisition details, consistent with SARProcessor's
        _save_results pattern."""
        metadata_dir = f"dataset/{self.region}/metadata"
        os.makedirs(metadata_dir, exist_ok=True)
        json_path = Path(metadata_dir) / f"{self.region}_dem.json"

        entry = {
            "source": "COPERNICUS/DEM/GLO30",
            "scale_m": scale,
            "crs": crs,
            "method": method,
            "output_path": str(output_path),
            "acquired_at": datetime.now().isoformat(),
        }

        if json_path.exists():
            with open(json_path, "r") as f:
                existing_data = json.load(f)
                if isinstance(existing_data, list):
                    existing_data.append(entry)
                else:
                    existing_data = [existing_data, entry]
        else:
            existing_data = [entry]

        with open(json_path, "w") as f:
            json.dump(existing_data, f, indent=2)

        print(f"  📝 DEM metadata saved to {json_path}")

    def get_elevation_stats(self, dem_path=None):
        """
        Quick sanity check on a downloaded DEM -- min/max/mean elevation.
        Useful to confirm the download covers real terrain (e.g. not all
        zeros or nodata) before using it for volume calculations.
        """
        if dem_path is None:
            dem_path = Path(
                f"dataset/{self.region}/raw/dem/{self.region}_dem_glo30.tif"
            )

  ⬇️ Downloading Copernicus GLO-30 DEM for bogoria...
Generating URL ...
Please wait ...
Data downloaded to /home/desy/rift-waters/notebooks/dataset/bogoria/raw/dem/bogoria_dem_glo30.tif
  ✅ Saved DEM to dataset/bogoria/raw/dem/bogoria_dem_glo30.tif
  📝 DEM metadata saved to dataset/bogoria/metadata/bogoria_dem.json
  📊 Elevation range: 990.8m - 2251.6m (mean 1311.6m)


{'min_m': 990.77392578125,
 'max_m': 2251.58154296875,
 'mean_m': 1311.605224609375}

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show


water_mask = "/home/desy/rift-waters/notebooks/dataset/bogoria/raw/dem/bogoria_dem_glo30.tif"

with rasterio.open(water_mask) as src:
    image = src.read(1)
    print(image.shape)
    print(image.min(), image.max())
    plt.imshow(image, cmap='terrain')
    plt.colorbar()
    plt.show()

IndexError: band index 2 out of range (not in (1,))

In [ ]:
gdf.explore(color='cyan', tiles='CartoDB positron')

ImportError: The 'folium>=0.12', 'matplotlib' and 'mapclassify' packages are required for 'explore()'. You can install them using 'conda install -c conda-forge "folium>=0.12" matplotlib mapclassify' or 'pip install "folium>=0.12" matplotlib mapclassify'.